Mapping particle tracks from Parcels unto the Salish Sea Atlantis Boxes. 

Original code written by Bec Gordon & Javier Porobic, CSIRO.
Link to the [SSAM Ocean Parcels Repo](https://bitbucket.csiro.au/users/por07g/repos/ssam_oceanparcels/browse)

This notebook includes pseudo-components of PAHs, grouped by log(k_{ow}) as follows: Naphthalene (log(Kow) 3.2 - 3.5) = Naphthalene (C0-C4)
Acenaphthylene; Phenanthrene (log(Kow) 3.8 - 4.6) = Phenanthrene (C0-C4), Acenaphthene, Fluorene, Anthracene; Pyrene(log(Kow) 4.9 - 5.2) = Pyrene, Fluoranthene; Benzo (log(Kow) 5.7 - 6.5) = Chrysene, benzo(e)pyrene, benzo(a)pyrene, perylene, Indeno(1,2,3-cd)pyrene, Benz(a)anthracene, Benzo(b)fluoranthene, Benzo(k)fluoranthene. 

In [1]:
import os
import xarray as xr
import numpy as np
import geopandas as gpd
import pandas as pd
from netCDF4 import Dataset
import matplotlib.pyplot as plt
from shapely.geometry import MultiPoint

In [2]:
shapefile_name = "/ocean/rlovindeer/Atlantis/ssam_oceanparcels/SalishSea/SalishSea_July172019_2/SalishSea_July172019.shp"
data_df_original = gpd.read_file(shapefile_name)
data_df_original = data_df_original.sort_values(by=['BOX_ID'])
data_df = data_df_original.set_index('BOX_ID')
box_depth = data_df['BOTZ']
box_area = data_df['AREA']

In [3]:
surface_layer = []
for box in range(0,130):
    if data_df.iloc[box].BOTZ < 26:
        layer = 0
    elif data_df.iloc[box].BOTZ == 50:
        layer = 1
    elif data_df.iloc[box].BOTZ == 100:
        layer = 2
    elif data_df.iloc[box].BOTZ == 200:
        layer = 3
    elif data_df.iloc[box].BOTZ > 200 and box_depth[box] < 401:
        layer = 4
    elif data_df.iloc[box].BOTZ > 400:
        layer = 5
    surface_layer.append(layer)

In [4]:
len(surface_layer)

130

In [65]:
# Ocean Parcels Spill File

inputFileName = '6a_VancouverHarbour_BunkerC_2020-07-13' #summer: ; winter: 
num_particles = 10000
file_path = '/ocean/rlovindeer/MOAD/analysis-raisha/notebooks/contaminant-dispersal/results/' + inputFileName + '_'+ str(num_particles) +'_OP_D50_wp3.zarr'
print('Running '+inputFileName)

Running 6a_VancouverHarbour_BunkerC_2020-07-13


In [66]:
scenario = inputFileName.split(sep = '_')

In [67]:
# Oil type properties & spill location selection

Dilbit = {
    "Density": 984.6, #kg/m^3 17% mass loss
    "Naphthalene": 627, #mg/kg oil
    "Phenanthrene": 1396,
    "Pyrene": 15,
    "Benzo": 279,
}

BunkerC = {
    "Density": 1017.8, #10% mass loss
    "Naphthalene": 15767,
    "Phenanthrene": 20373,
    "Pyrene": 381,
    "Benzo": 4608,
}

Diesel = {
    "Density": 841.6, # 22% mass loss
    "Naphthalene": 24378,
    "Phenanthrene": 6248,
    "Pyrene": 47,
    "Benzo": 0,
}

WSFmean = {
    "Naphthalene": 0.47,
    "Phenanthrene": 0.093,
    "Pyrene": 0.0035,
    "Benzo": 0.0035,
}

WSFmax = {
    "Naphthalene": 0.79,
    "Phenanthrene": 0.15,
    "Pyrene": 0.006,
    "Benzo": 0.006,
}

WSFmin = {
    "Naphthalene": 0.28,
    "Phenanthrene": 0.061,
    "Pyrene": 0.0021,
    "Benzo": 0.0021,
}

fuel_type = {
    "Dilbit" : Dilbit,
    "BunkerC" : BunkerC,
    "Diesel" : Diesel,
}

spill_volume = {
    "5b" : 2000, #m^3 
    "6a" : 15,
    "7a" : 1000,
    "4a" : 500,
}

spill_box_surface_volume = {
    "5b" : (322271112.331102*25), #m^3 area x surface depth
    "6a" : (108463283.03614*25),
    "7a" : (663754967.760742*25),
    "4a" : ((289374380+143789739)/2*25),
}

In [68]:
# Calculations of oil per particle in mg/m^3/particle
release_start = scenario[3]
oil_per_particle = (fuel_type[scenario[2]]["Density"] * spill_volume[scenario[0]] / spill_box_surface_volume[scenario[0]]) / num_particles #kg/m3
naph_per_particle = oil_per_particle * fuel_type[scenario[2]]["Naphthalene"] * WSFmean["Naphthalene"] #mg/m3
phen_per_particle = oil_per_particle * fuel_type[scenario[2]]["Phenanthrene"] * WSFmean["Phenanthrene"]
pyrene_per_particle = oil_per_particle * fuel_type[scenario[2]]["Pyrene"] * WSFmean["Pyrene"]
benzo_per_particle = oil_per_particle * fuel_type[scenario[2]]["Benzo"] * WSFmean["Benzo"]
tpah_per_particle = naph_per_particle + phen_per_particle + pyrene_per_particle + benzo_per_particle
release_start_time = np.datetime64(release_start)

In [69]:
# Calculations of oil mass in mg
oil_mass_kg = (fuel_type[scenario[2]]["Density"] * spill_volume[scenario[0]])
naph_mass_mg = oil_mass_kg * fuel_type[scenario[2]]["Naphthalene"] / 1e6
phen_mass_mg = oil_mass_kg * fuel_type[scenario[2]]["Phenanthrene"] / 1e6
pyrene_mass_mg = oil_mass_kg * fuel_type[scenario[2]]["Pyrene"] / 1e6
benzo_mass_mg = oil_mass_kg * fuel_type[scenario[2]]["Benzo"] / 1e6

print(str(scenario[2])+' spill mass of '+str(oil_mass_kg)+' in kg')
print('Total Naphthalene spill mass of '+str(naph_mass_mg)+' in kg')
print('Total Phenanthrene spill mass of '+str(phen_mass_mg)+' in kg')
print('Total Pyrene spill mass of '+str(pyrene_mass_mg)+' in kg')
print('Total Benzo spill mass of '+str(benzo_mass_mg)+' in kg')

BunkerC spill mass of 15267.0 in kg
Total Naphthalene spill mass of 240.714789 in kg
Total Phenanthrene spill mass of 311.034591 in kg
Total Pyrene spill mass of 5.816727 in kg
Total Benzo spill mass of 70.350336 in kg


In [70]:
# Calculating the surface concentration of each contaminant for the entire spill
oil_full = oil_per_particle * num_particles * 1e6
naph_full = naph_per_particle * num_particles
phen_full = phen_per_particle * num_particles
pyrene_full = pyrene_per_particle * num_particles
benzo_full = benzo_per_particle * num_particles
tpah_full = tpah_per_particle * num_particles

print(str(scenario[2])+' spill concentration = '+str(oil_full)+' in mg/m^3')
print('Total PAH water soluble fraction = '+str(tpah_full)+' in mg/m3')
print('Naphthalene water soluble fraction = '+str(naph_full)+' in mg/m^3')
print('Phenanthrene water soluble fraction '+str(phen_full)+' in mg/m^3')
print('Pyrene water soluble fraction '+str(pyrene_full)+' in mg/m^3')
print('Benzo water soluble fraction '+str(benzo_full)+' in mg/m^3')

BunkerC spill concentration = 5.63029241698798 in mg/m^3
Total PAH water soluble fraction = 0.05248919211345503 in mg/m3
Naphthalene water soluble fraction = 0.04172322565316525 in mg/m^3
Phenanthrene water soluble fraction 0.010667653109250538 in mg/m^3
Pyrene water soluble fraction 7.5079949380534715e-06 in mg/m^3
Benzo water soluble fraction 9.080535610118213e-05 in mg/m^3


In [71]:
numLayers = 7
numBoxes = data_df.shape[0]
outputDT = 43100.00
print('numBoxes = ' + str(numBoxes))

numBoxes = 130


In [72]:
pfile = xr.open_zarr(file_path) #.to_netcdf(new_file_name)

In [73]:
#an array of times that correspond to Atlantis timesteps
time_slice = np.arange(0,len(pfile.time[0]),12) #or any N

In [74]:
time_slice

array([  0,  12,  24,  36,  48,  60,  72,  84,  96, 108, 120, 132, 144,
       156, 168, 180, 192, 204, 216, 228, 240])

In [75]:
reduced_pfile = pfile.isel(obs=(time_slice))

In [76]:
reduced_pfile

<xarray.Dataset>
Dimensions:      (trajectory: 10000, obs: 21)
Coordinates:
  * obs          (obs) int32 0 12 24 36 48 60 72 ... 168 180 192 204 216 228 240
  * trajectory   (trajectory) int64 1 81 100 116 173 ... 9926 9935 9971 9984
Data variables:
    age          (trajectory, obs) float64 dask.array<chunksize=(207, 1), meta=np.ndarray>
    beached      (trajectory, obs) float64 dask.array<chunksize=(207, 1), meta=np.ndarray>
    decay_value  (trajectory, obs) float32 dask.array<chunksize=(207, 1), meta=np.ndarray>
    lat          (trajectory, obs) float64 dask.array<chunksize=(207, 1), meta=np.ndarray>
    lon          (trajectory, obs) float64 dask.array<chunksize=(207, 1), meta=np.ndarray>
    time         (trajectory, obs) datetime64[ns] dask.array<chunksize=(207, 1), meta=np.ndarray>
    z            (trajectory, obs) float64 dask.array<chunksize=(207, 1), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_mesh:           spherical
    parcels_version:        2.4.2

In [77]:
# pulling all variables from Parcels
lon = reduced_pfile.variables['lon']
time = reduced_pfile.variables['time']

In [78]:
numParticles = lon.shape[0]
numSteps = len(time[0,:])

print('numParticles = ' + str(numParticles))
print('numSteps = ' + str(numSteps))

numParticles = 10000
numSteps = 21


In [79]:
minDate = np.datetime64(release_start+"T00:30:00")
ts = pd.to_datetime(str(minDate))
d = ts.strftime('%Y-%m-%d %H:%M:%S')
print(d)

2020-07-13 00:30:00


In [80]:
# Create the netcdf output file

netcdfFilePath = "/ocean/rlovindeer/MOAD/analysis-raisha/notebooks/contaminant-dispersal/results/ForcingFiles/"
netcdfFileName = netcdfFilePath+"SSAM_Scenario_"+scenario[0]+"_"+ scenario[3] + "_" + str(num_particles) + "_WSFmean.nc"
try:
    os.remove(netcdfFileName)
except:
    pass
ncfile = Dataset(netcdfFileName, "w", format="NETCDF4", clobber=True)
Dataset.set_fill_on(ncfile)

# Dimensions
t = ncfile.createDimension("t", None)
b = ncfile.createDimension("b", numBoxes)
z = ncfile.createDimension("z", numLayers)

In [81]:
# Variables
times = ncfile.createVariable("t",np.float64, ("t",))
TPAH = ncfile.createVariable("TPAH",np.float64,("t", "b"))
Naphthalene = ncfile.createVariable("Naphthalene",np.float64, ("t", "b", "z"))
Phenanthrene = ncfile.createVariable("Phenanthrene",np.float64,("t", "b", "z"))
Pyrene = ncfile.createVariable("Pyrene",np.float64,("t", "b", "z"))
Benzo = ncfile.createVariable("Benzo",np.float64,("t", "b", "z"))

# Attributes
Naphthalene.units = "mgPAH/m^3"
Naphthalene.long_name = "Naphthalene"
Naphthalene.missing_value = 0.0000
Naphthalene.valid_min = 0.0000
Naphthalene.valid_max = 100000000.0

Phenanthrene.units = "mgPAH/m^3"
Phenanthrene.long_name = "Phenanthrene"
Phenanthrene.missing_value = 0.0000
Phenanthrene.valid_min = 0.0000
Phenanthrene.valid_max = 100000000.0

Pyrene.units = "mgPAH/m^3"
Pyrene.long_name = "Pyrene"
Pyrene.missing_value = 0.0000
Pyrene.valid_min = 0.0000
Pyrene.valid_max = 100000000.0

Benzo.units = "mgPAH/m^3"
Benzo.long_name = "Benzo(a)pyrene"
Benzo.missing_value = 0.0000
Benzo.valid_min = 0.0000
Benzo.valid_max = 100000000.0

TPAH.units = "mgPAH/m^3"
TPAH.long_name = "Total Water Soluble Fraction PAH Concentration"

times.units = "seconds since " + d
times.dt = outputDT
times.long_name = "time"

In [82]:
# Populate variables with contaminant data

timeData = np.arange(0,(len(time[0,:]))*outputDT,outputDT)
times[:] = timeData

No_layer_particles = np.zeros((numSteps, numBoxes))
Surface_particles = np.zeros((numSteps, numBoxes, numLayers))

In [83]:
for timeValue in range(0, numSteps):

    lats = reduced_pfile['lat'][:,timeValue].values
    lons = reduced_pfile['lon'][:,timeValue].values
    points = np.column_stack((lons, lats))
    multi_point = MultiPoint(points)

    for box_number in range (0, numBoxes):
        layer = surface_layer[box_number]
        box_coordinates = data_df.iloc[box_number].geometry
        found_particles = np.array([box_coordinates.contains(point) for point in multi_point])
        num_particles_within_box = np.sum(found_particles)
        Surface_particles[timeValue][box_number][layer] = Surface_particles[timeValue][box_number][layer] + num_particles_within_box
        No_layer_particles[timeValue][box_number] = No_layer_particles[timeValue][box_number] + num_particles_within_box

Naphthalene[:,:,:] = Surface_particles * naph_per_particle
Phenanthrene[:,:,:] = Surface_particles * phen_per_particle
Pyrene[:,:,:] = Surface_particles * pyrene_per_particle
Benzo[:,:,:] = Surface_particles * benzo_per_particle
TPAH[:,:] = No_layer_particles * tpah_per_particle

ncfile.close()

In [84]:
np.histogram(No_layer_particles)

(array([2711,   17,    0,    0,    0,    1,    0,    0,    0,    1]),
 array([    0.,  1000.,  2000.,  3000.,  4000.,  5000.,  6000.,  7000.,
         8000.,  9000., 10000.]))